<a href="https://colab.research.google.com/github/ujwalbajaj/cd/blob/main/CD_Assignments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LEX and YACC Compiler in Colab

Drawbacks:
* Regular interrupts (Ctrl+D, Ctrl+C) for shell won't work in Colab while inputting for program.
<br>Workaround: Store your inputs in a txt file and pass it to the program.

In [28]:
#@title Install *prerqeuisites* (run this cell first to work on LEX/YACC)
!sudo apt install flex bison

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
flex is already the newest version (2.6.4-8.2build1).
bison is already the newest version (2:3.8.2+dfsg-1ubuntu0.24.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


## Scientific Calculator (Lex & Yacc)

## Lex and Yacc combined

In [29]:
%%writefile program.y

%{
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

int yylex(void);
void yyerror(const char *s);
%}

%union {
    double value;
}

%token <value> NUMBER
%token SIN COS TAN SQRT LOG EXP POW INVALID
%type <value> expression

%left '+' '-'
%left '*' '/'
%right UMINUS

%%

input:
      expression '\n' {
          printf("Result = %.4lf\n", $1);
      }
    | expression {
          printf("Result = %.4lf\n", $1);
      }
    ;

expression:
      expression '+' expression {
          $$ = $1 + $3;
      }
    | expression '-' expression {
          $$ = $1 - $3;
      }
    | expression '*' expression {
          $$ = $1 * $3;
      }
    | expression '/' expression {
          if ($3 == 0) {
              yyerror("Division by zero");
              YYABORT;
          }
          $$ = $1 / $3;
      }
    | '-' expression %prec UMINUS {
          $$ = -$2;
      }
    | '(' expression ')' {
          $$ = $2;
      }
    | SIN '(' expression ')' {
          $$ = sin($3);
      }
    | COS '(' expression ')' {
          $$ = cos($3);
      }
    | TAN '(' expression ')' {
          $$ = tan($3);
      }
    | SQRT '(' expression ')' {
          if ($3 < 0) {
              yyerror("Invalid input for square root");
              YYABORT;
          }
          $$ = sqrt($3);
      }
    | LOG '(' expression ')' {
          if ($3 <= 0) {
              yyerror("Invalid input for logarithm");
              YYABORT;
          }
          $$ = log($3);
      }
    | EXP '(' expression ')' {
          $$ = exp($3);
      }
    | POW '(' expression ',' expression ')' {
          $$ = pow($3, $5);
      }
    | NUMBER {
          $$ = $1;
      }
    | INVALID {
          yyerror("Invalid character");
          YYABORT;
      }
    ;

%%

void yyerror(const char *s)
{
    printf("Error: %s\n", s);
}

int main(void)
{
    printf("SCIENTIFIC CALCULATOR\n");
    printf("=====================\n");
    yyparse();
    return 0;
}

Overwriting program.y


In [30]:
#@title Writing Lex program
%%writefile program.l

%{
#include "y.tab.h"
#include <stdlib.h>
%}

%%

[0-9]+(\.[0-9]+)?  { yylval.value = atof(yytext); return NUMBER; }

"sin"       { return SIN; }
"cos"       { return COS; }
"tan"       { return TAN; }
"sqrt"      { return SQRT; }
"log"       { return LOG; }
"exp"       { return EXP; }
"pow"       { return POW; }

"+"         { return '+'; }
"-"         { return '-'; }
"*"         { return '*'; }
"/"         { return '/'; }
"("         { return '('; }
")"         { return ')'; }
","         { return ','; }

[ \t\r]+    { }
\n          { return '\n'; }

.           { return INVALID; }

%%

int yywrap()
{
    return 1;
}

Overwriting program.l


if you want to use at txt as an input

In [31]:
%%writefile program.txt
2+3*4


Overwriting program.txt


In [32]:
#@title Shell Execution
%%shell

yacc -d program.y
lex program.l
gcc y.tab.c lex.yy.c -lm -lfl
./a.out < program.txt

SCIENTIFIC CALCULATOR
Result = 14.0000


## While Loop

### 1. YACC Specification (`while_parser.y`)
This file defines the grammar rules for parsing a valid 'C' `while` loop syntax.

In [33]:
%%writefile while_parser.y

%{
#include <stdio.h>
#include <stdlib.h>

int yylex(void);
void yyerror(const char *s);
%}

%token WHILE ID NUM EQ NE LE GE AND OR

%right '='
%left OR
%left AND
%left EQ NE '<' '>' LE GE
%left '+' '-'
%left '*' '/'

%%

program:
    while_stmt { printf("Valid 'while' statement syntax!\n"); exit(0); }
    ;

while_stmt:
    WHILE '(' condition ')' block
    ;

condition:
    expression rel_op expression
    | expression
    ;

rel_op:
    '<' | '>' | LE | GE | EQ | NE
    ;

expression:
    expression '+' expression
    | expression '-' expression
    | expression '*' expression
    | expression '/' expression
    | ID '=' expression
    | ID
    | NUM
    | '(' expression ')'
    ;

block:
    '{' statement_list '}'
    | statement ';'
    ;

statement_list:
    statement_list statement ';'
    | /* empty */
    ;

statement:
    expression
    | while_stmt
    ;

%%

void yyerror(const char *s) {
    fprintf(stderr, "Syntax Error: %s\n", s);
    exit(1);
}

int main(void) {
    printf("Enter 'while' statement to check syntax:\n");
    yyparse();
    return 0;
}

Writing while_parser.y


### 2. LEX Specification (`while_lexer.l`)
This file tokenizes variables, numbers, operators, and the `while` keyword.

In [34]:
%%writefile while_lexer.l

%{
#include "y.tab.h"
%}

%%

"while"                 { return WHILE; }
[a-zA-Z_][a-zA-Z0-9_]*  { return ID; }
[0-9]+                  { return NUM; }

"=="                    { return EQ; }
"!="                    { return NE; }
"<="                    { return LE; }
">="                    { return GE; }
"&&"                    { return AND; }
"||"                    { return OR; }

[ \t\n\r]+              { /* ignore whitespace */ }
.                       { return yytext[0]; }

%%

int yywrap() {
    return 1;
}

Writing while_lexer.l


### 3. Test Cases (`while_input.txt`)
Below are sample tests representing a correct 'C' language `while` loop syntax structure.

In [35]:
%%writefile while_input.txt
while (x < 10) {
    x = x + 1;
}

Writing while_input.txt


### 4. Compilation & Execution
This compiles our lexer and parser and validates the code in `while_input.txt`.

In [36]:
%%shell
yacc -d while_parser.y
lex while_lexer.l
gcc y.tab.c lex.yy.c -lfl
./a.out < while_input.txt

Enter 'while' statement to check syntax:
Valid 'while' statement syntax!


### Common Subexpression Elimination (CSE) Optimization Demo
Common Subexpression Elimination is an optimization where identical expressions that evaluate to the same value are identified and replaced with a single variable holding the computed value.

In [37]:
def common_subexpression_elimination(instructions):
    optimized = []
    expr_map = {}

    for instr in instructions:
        # Expecting three-address code instructions: ['dst', '=', 'arg1', 'op', 'arg2']
        if len(instr) == 5 and instr[1] == '=':
            dst, _, arg1, op, arg2 = instr
            expr = (arg1, op, arg2)
            reverse_expr = (arg2, op, arg1) if op in ['+', '*'] else None

            if expr in expr_map:
                optimized.append([dst, '=', expr_map[expr]])
            elif reverse_expr and reverse_expr in expr_map:
                optimized.append([dst, '=', expr_map[reverse_expr]])
            else:
                expr_map[expr] = dst
                optimized.append(instr)
        else:
            optimized.append(instr)

    return optimized

# Sample Three-Address Code (TAC)
tac_code = [
    ['t1', '=', 'a', '*', 'b'],
    ['t2', '=', 'c', '+', 'd'],
    ['t3', '=', 'a', '*', 'b'], # Redundant subexpression
    ['t4', '=', 't2', '+', 't3']
]

optimized_tac = common_subexpression_elimination(tac_code)

print("Original TAC:")
for line in tac_code: print(' '.join(line))

print("\nOptimized TAC (after CSE):")
for line in optimized_tac: print(' '.join(line))

Original TAC:
t1 = a * b
t2 = c + d
t3 = a * b
t4 = t2 + t3

Optimized TAC (after CSE):
t1 = a * b
t2 = c + d
t3 = t1
t4 = t2 + t3


## Common Subexpression Elimination

### C++ Implementation of Common Subexpression Elimination (CSE)
Below is the complete C++ compiler optimization source code to eliminate duplicate expressions from Three-Address Code.

In [40]:
%%writefile cse_parser.y

%{
#include <iostream>
#include <vector>
#include <string>
#include <map>
#include <tuple>
#include <cstring>

using namespace std;

extern int yylex();
extern int yylineno;
void yyerror(const char *s);

struct Instruction {
    string dst;
    string arg1;
    string op;
    string arg2;
};

vector<Instruction> original_tac;
vector<Instruction> optimized_tac;
map<tuple<string, string, string>, string> expr_map;
int temp_count = 1;

string new_temp() {
    return "t" + to_string(temp_count++);
}

char* value_of(string s) {
    char* res = new char[s.length() + 1];
    strcpy(res, s.c_str());
    return res;
}

string add_tac(string arg1, string op, string arg2) {
    string dst = new_temp();
    original_tac.push_back({dst, arg1, op, arg2});

    auto expr = make_tuple(arg1, op, arg2);
    auto reverse_expr = make_tuple(arg2, op, arg1);
    bool is_commutative = (op == "+" || op == "*");

    if (expr_map.find(expr) != expr_map.end()) {
        optimized_tac.push_back({dst, expr_map[expr], "", ""});
    } else if (is_commutative && expr_map.find(reverse_expr) != expr_map.end()) {
        optimized_tac.push_back({dst, expr_map[reverse_expr], "", ""});
    } else {
        expr_map[expr] = dst;
        optimized_tac.push_back({dst, arg1, op, arg2});
    }
    return dst;
}
%}

%union {
    char* str;
}

%token <str> NUM ID
%type <str> expression

%left '+' '-'
%left '*' '/'

%%

program:
    statements
    ;

statements:
    statements statement
    | /* empty */
    ;

statement:
    ID '=' expression '\n' {
        original_tac.push_back({$1, $3, "", ""});
        optimized_tac.push_back({$1, $3, "", ""});
    }
    | expression '\n'
    ;

expression:
    expression '+' expression { $$ = value_of(add_tac($1, "+", $3)); }
    | expression '-' expression { $$ = value_of(add_tac($1, "-", $3)); }
    | expression '*' expression { $$ = value_of(add_tac($1, "*", $3)); }
    | expression '/' expression { $$ = value_of(add_tac($1, "/", $3)); }
    | '(' expression ')'        { $$ = $2; }
    | NUM                       { $$ = $1; }
    | ID                        { $$ = $1; }
    ;

%%

void yyerror(const char *s) {
    cerr << "Error on line " << yylineno << ": " << s << endl;
}

int main() {
    yyparse();

    cout << "=== Original TAC ===" << endl;
    for (const auto& instr : original_tac) {
        if (instr.op == "") cout << instr.dst << " = " << instr.arg1 << endl;
        else cout << instr.dst << " = " << instr.arg1 << " " << instr.op << " " << instr.arg2 << endl;
    }

    cout << "\n=== Optimized TAC (after CSE) ===" << endl;
    for (const auto& instr : optimized_tac) {
        if (instr.op == "") cout << instr.dst << " = " << instr.arg1 << endl;
        else cout << instr.dst << " = " << instr.arg1 << " " << instr.op << " " << instr.arg2 << endl;
    }
    return 0;
}

Writing cse_parser.y


In [39]:
%%shell
g++ -O3 cse_optimization.cpp -o cse_opt
./cse_opt

Original TAC:
t1 = a * b
t2 = c + d
t3 = a * b
t4 = t2 + t3

Optimized TAC (after CSE):
t1 = a * b
t2 = c + d
t3 = t1
t4 = t2 + t3


### Dynamic YACC & LEX CSE Optimizer
This implementation parses mathematical expressions, generates Three-Address Code (TAC), and automatically applies Common Subexpression Elimination (CSE) optimizations in C++ structures within the Yacc parser actions.

In [45]:
%%writefile cse_parser.y

%{
#include <iostream>
#include <vector>
#include <string>
#include <map>
#include <tuple>
#include <cstring>

using namespace std;

extern int yylex();
extern int yylineno;
void yyerror(const char *s);

struct Instruction {
    string dst;
    string arg1;
    string op;
    string arg2;
};

vector<Instruction> original_tac;
vector<Instruction> optimized_tac;
map<tuple<string, string, string>, string> expr_map;
int temp_count = 1;

string new_temp() {
    return "t" + to_string(temp_count++);
}

char* value_of(string s) {
    char* res = new char[s.length() + 1];
    strcpy(res, s.c_str());
    return res;
}

string add_tac(string arg1, string op, string arg2) {
    string dst = new_temp();
    original_tac.push_back({dst, arg1, op, arg2});

    auto expr = make_tuple(arg1, op, arg2);
    auto reverse_expr = make_tuple(arg2, op, arg1);
    bool is_commutative = (op == "+" || op == "*");

    if (expr_map.find(expr) != expr_map.end()) {
        optimized_tac.push_back({dst, expr_map[expr], "", ""});
    } else if (is_commutative && expr_map.find(reverse_expr) != expr_map.end()) {
        optimized_tac.push_back({dst, expr_map[reverse_expr], "", ""});
    } else {
        expr_map[expr] = dst;
        optimized_tac.push_back({dst, arg1, op, arg2});
    }
    return dst;
}
%}

%union {
    char* str;
}

%token <str> NUM ID
%type <str> expression

%left '+' '-'
%left '*' '/'

%%

program:
    statements
    ;

statements:
    statements statement
    | /* empty */
    ;

statement:
    ID '=' expression '\n' {
        original_tac.push_back({$1, $3, "", ""});
        optimized_tac.push_back({$1, $3, "", ""});
    }
    | expression '\n'
    ;

expression:
    expression '+' expression { $$ = value_of(add_tac($1, "+", $3)); }
    | expression '-' expression { $$ = value_of(add_tac($1, "-", $3)); }
    | expression '*' expression { $$ = value_of(add_tac($1, "*", $3)); }
    | expression '/' expression { $$ = value_of(add_tac($1, "/", $3)); }
    | '(' expression ')'        { $$ = $2; }
    | NUM                       { $$ = $1; }
    | ID                        { $$ = $1; }
    ;

%%

void yyerror(const char *s) {
    cerr << "Error on line " << yylineno << ": " << s << endl;
}

int main() {
    yyparse();

    cout << "=== Original TAC ===" << endl;
    for (const auto& instr : original_tac) {
        if (instr.op == "") cout << instr.dst << " = " << instr.arg1 << endl;
        else cout << instr.dst << " = " << instr.arg1 << " " << instr.op << " " << instr.arg2 << endl;
    }

    cout << "\n=== Optimized TAC (after CSE) ===" << endl;
    for (const auto& instr : optimized_tac) {
        if (instr.op == "") cout << instr.dst << " = " << instr.arg1 << endl;
        else cout << instr.dst << " = " << instr.arg1 << " " << instr.op << " " << instr.arg2 << endl;
    }
    return 0;
}

Overwriting cse_parser.y


In [42]:
%%writefile cse_lexer.l

%{
#include "y.tab.h"
#include <string.h>
%}

%option yylineno

%%

[0-9]+                  { yylval.str = strdup(yytext); return NUM; }
[a-zA-Z_][a-zA-Z0-9_]*  { yylval.str = strdup(yytext); return ID; }

[ \t]                   { /* ignore whitespace */ }

.                       { return yytext[0]; }

%%

int yywrap() {
    return 1;
}

Writing cse_lexer.l


In [49]:
%%writefile cse_input.txt
x = a * b + c + a * b

Overwriting cse_input.txt


In [50]:
%%shell
yacc -d cse_parser.y
lex cse_lexer.l
g++ -w y.tab.c lex.yy.c -o cse_compiler
./cse_compiler < cse_input.txt


Error on line 1: syntax error
=== Original TAC ===
t1 = a * b
t2 = t1 + c
t3 = a * b
t4 = t2 + t3

=== Optimized TAC (after CSE) ===
t1 = a * b
t2 = t1 + c
t3 = t1
t4 = t2 + t3


## Optimization of input Quadruples

In [51]:
%%writefile constant_folding.cpp
#include <iostream>
#include <vector>
#include <string>
#include <unordered_map>
#include <sstream>
#include <cctype>

using namespace std;

struct Quadruple {
    string op;
    string arg1;
    string arg2;
    string result;
};

bool is_number(const string& s) {
    if (s.empty()) return false;
    size_t start = (s[0] == '-' || s[0] == '+') ? 1 : 0;
    if (start == s.size()) return false;
    for (size_t i = start; i < s.size(); ++i) {
        if (!isdigit(s[i])) return false;
    }
    return true;
}

int perform_op(int a, const string& op, int b) {
    if (op == "+") return a + b;
    if (op == "-") return a - b;
    if (op == "*") return a * b;
    if (op == "/") return (b != 0) ? (a / b) : 0;
    return 0;
}

int main() {
    vector<Quadruple> original = {
        {"*", "3", "4", "t1"},      // t1 = 3 * 4
        {"+", "a", "t1", "t2"},     // t2 = a + t1
        {"/", "10", "2", "t3"},     // t3 = 10 / 2
        {"+", "t2", "t3", "t4"}      // t4 = t2 + t3
    };

    unordered_map<string, string> constant_map;
    vector<Quadruple> optimized;

    for (const auto& q : original) {
        string a1 = q.arg1;
        string a2 = q.arg2;

        if (constant_map.count(a1)) a1 = constant_map[a1];
        if (constant_map.count(a2)) a2 = constant_map[a2];

        if (is_number(a1) && is_number(a2)) {
            int res_val = perform_op(stoi(a1), q.op, stoi(a2));
            constant_map[q.result] = to_string(res_val);
            optimized.push_back({"=", to_string(res_val), "", q.result});
        } else {
            optimized.push_back({q.op, a1, a2, q.result});
        }
    }

    cout << "=== Original Quadruples ===" << endl;
    for (const auto& q : original) {
        cout << q.result << " = " << q.arg1 << " " << q.op << " " << q.arg2 << endl;
    }

    cout << "\n=== Optimized Quadruples (Constant Folding) ===" << endl;
    for (const auto& q : optimized) {
        if (q.op == "=") {
            cout << q.result << " = " << q.arg1 << endl;
        } else {
            cout << q.result << " = " << q.arg1 << " " << q.op << " " << q.arg2 << endl;
        }
    }
    return 0;
}

Writing constant_folding.cpp


In [52]:
%%shell
g++ constant_folding.cpp -o constant_folding
./constant_folding

=== Original Quadruples ===
t1 = 3 * 4
t2 = a + t1
t3 = 10 / 2
t4 = t2 + t3

=== Optimized Quadruples (Constant Folding) ===
t1 = 12
t2 = a + 12
t3 = 5
t4 = t2 + 5


In [53]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.3 MB/s eta 0:00:00


In [54]:
import subprocess
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors

# Run the binary to catch the exact outputs dynamically
result = subprocess.run(['./constant_folding'], capture_output=True, text=True)
output_text = result.stdout

# Read source code
with open('constant_folding.cpp', 'r') as f:
    source_code = f.read()

doc = SimpleDocTemplate("constant_folding_report.pdf", pagesize=letter,
                        rightMargin=40, leftMargin=40, topMargin=40, bottomMargin=40)

styles = getSampleStyleSheet()
title_style = ParagraphStyle(
    'TitleStyle',
    parent=styles['Heading1'],
    fontSize=22,
    textColor=colors.HexColor('#1A365D'),
    spaceAfter=15
)

section_style = ParagraphStyle(
    'SectionStyle',
    parent=styles['Heading2'],
    fontSize=14,
    textColor=colors.HexColor('#2B6CB0'),
    spaceBefore=12,
    spaceAfter=6
)

code_style = ParagraphStyle(
    'CodeStyle',
    parent=styles['Normal'],
    fontName='Courier',
    fontSize=9,
    leading=11,
    textColor=colors.HexColor('#2D3748'),
    backColor=colors.HexColor('#F7FAFC'),
    borderColor=colors.HexColor('#E2E8F0'),
    borderWidth=1,
    borderPadding=8,
    spaceAfter=15
)

body_style = ParagraphStyle(
    'BodyStyle',
    parent=styles['Normal'],
    fontSize=10,
    leading=14,
    spaceAfter=10
)

story = []

# Page 1: Title and Input / Output
story.append(Paragraph("Compiler Optimization: Constant Folding", title_style))
story.append(Paragraph("This document demonstrates the implementation of <b>Constant Folding</b> inside a compiler toolchain backend structure.", body_style))
story.append(Spacer(1, 10))

story.append(Paragraph("1. Input Quadruples Set", section_style))
input_quads = """t1 = 3 * 4\nt2 = a + t1\nt3 = 10 / 2\nt4 = t2 + t3"""
story.append(Paragraph(input_quads.replace('\n', '<br/>'), code_style))

story.append(Paragraph("2. Execution Output (Original vs Optimized)", section_style))
story.append(Paragraph(output_text.replace('\n', '<br/>'), code_style))

story.append(PageBreak())

# Page 2: C++ Source Code
story.append(Paragraph("3. C++ Source Code", section_style))
# Escape HTML characters for safely inserting into reportlab Paragraph
escaped_source = source_code.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;').replace('\n', '<br/>').replace(' ', '&nbsp;')
story.append(Paragraph(escaped_source, code_style))

doc.build(story)
print("PDF successfully generated: constant_folding_report.pdf")

PDF successfully generated: constant_folding_report.pdf


In [55]:
from google.colab import files
files.download('constant_folding_report.pdf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>